# Namma Space: Cloud GPU Smoke Test (Google Colab T4 / RunPod)

This notebook verifies the cloud GPU environment for 3D Gaussian Splatting:
- **Install Step**: Installs `uv`, creates a Python 3.10 environment at `/content/ns-env`, installs pinned PyTorch and Nerfstudio 1.1.5, and verifies CUDA in `ENV_PYTHON`.
- **Step 0: Probes & Full Tool Help Discovery**: Python, PyTorch, CUDA, nvcc, ninja, COLMAP CUDA audit, and full `--help` text.
- **Step 1: Environment & GPU Verification**
- **Step 2: COLMAP Camera Poses & Sparse Reconstruction** (supports CUDA and verified CPU fallback flags with `QT_QPA_PLATFORM=offscreen`)
- **Step 3a: Splatfacto 10-Iteration Dry-Run** (Verifies dataset loading and config.yml creation, timeout 1200s)
- **Step 3b: Splatfacto 2000-Iteration Training** (Preserves coordinate frame, viewer auto-quit enabled, timeout 2400s)
- **Step 4: Gaussian Splat PLY Export & Verification** (62 float32 properties)

All outputs and `smoke_report.json` are written to Google Drive inside a `try/finally` block to ensure persistence even on error.

In [ ]:
# Configuration & Inputs
# GITHUB_URL is ONLY for cloning the repository code.
# DRIVE_FRAMES_DIR is the ONLY input source for room frames (must contain >= 30 images).
GITHUB_URL = ""          # e.g. "https://github.com/your-org/namma-space.git"
DRIVE_FRAMES_DIR = ""    # e.g. "/content/drive/MyDrive/namma_space_runs/frames"

# Subsampling limit for smoke test (Requirement 3)
MAX_SMOKE_FRAMES = 100

# Timeouts in seconds (Requirements 5 & 6)
TIMEOUT_COLMAP_FEATURE_S = 1800 # 30 minutes for feature extraction
TIMEOUT_COLMAP_MATCH_S = 1800   # 30 minutes for matcher
TIMEOUT_COLMAP_MAP_S = 1800     # 30 minutes for mapper
TIMEOUT_TRAIN_3A_S = 1200       # 20 minutes for splatfacto dry-run (10 iterations)
TIMEOUT_TRAIN_3B_S = 2400       # 40 minutes for splatfacto training (2000 iterations)
TIMEOUT_EXPORT_S = 600          # 10 minutes for splat export


In [ ]:
# Mount Google Drive and Validate Input Configuration
import math
import os
import shutil
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RUN_DIR = Path('/content/drive/MyDrive/namma_space_runs')
    DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted at: {DRIVE_RUN_DIR}")
except ImportError:
    DRIVE_RUN_DIR = Path('./runs/cloud_smoke_test')
    DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Running in non-Colab environment. Saving to: {DRIVE_RUN_DIR}")

# Requirement 7: Input frames MUST come from Drive only (DRIVE_FRAMES_DIR).
# If frames are missing or folder has fewer than 30 images, stop with a clear message. Never pass None as a path.
if not DRIVE_FRAMES_DIR.strip():
    raise SystemExit("FATAL ERROR: DRIVE_FRAMES_DIR is not set. Frames must come from Drive only.")

FRAMES_DIR = Path(DRIVE_FRAMES_DIR.strip())
if not FRAMES_DIR.is_dir():
    raise SystemExit(f"FATAL ERROR: DRIVE_FRAMES_DIR directory does not exist: {FRAMES_DIR}")

all_valid_images = sorted([
    p for p in FRAMES_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in ('.jpg', '.jpeg', '.png')
])
if len(all_valid_images) < 30:
    raise SystemExit(
        f"FATAL ERROR: DRIVE_FRAMES_DIR must contain at least 30 images for COLMAP reconstruction. "
        f"Found {len(all_valid_images)} images at '{FRAMES_DIR}'."
    )

# Requirement 3: Select every k-th image into a working folder under the run directory
# so the smoke test never feeds more than MAX_SMOKE_FRAMES to COLMAP.
k = max(1, math.ceil(len(all_valid_images) / MAX_SMOKE_FRAMES))
selected_images = all_valid_images[::k][:MAX_SMOKE_FRAMES]

WORKING_FRAMES_DIR = DRIVE_RUN_DIR / "working_frames"
WORKING_FRAMES_DIR.mkdir(parents=True, exist_ok=True)

# Clean existing contents of working directory to ensure exact frame set
for old_f in WORKING_FRAMES_DIR.iterdir():
    if old_f.is_file() or old_f.is_symlink():
        old_f.unlink()

for img_p in selected_images:
    dst_p = WORKING_FRAMES_DIR / img_p.name
    try:
        dst_p.symlink_to(img_p.resolve())
    except Exception:
        shutil.copy2(img_p, dst_p)

print(
    f"Verified {len(all_valid_images)} input frames. Selected {len(selected_images)} frames "
    f"(every {k}-th frame, max {MAX_SMOKE_FRAMES}) into working directory: {WORKING_FRAMES_DIR}"
)

# Optional repository cloning if GITHUB_URL is provided
WORKSPACE_DIR = Path('/content/namma-space') if Path('/content').exists() else Path('./workspace')
if GITHUB_URL.strip():
    print(f"Cloning repository code from {GITHUB_URL}...")
    import subprocess
    if not WORKSPACE_DIR.exists():
        subprocess.run(["git", "clone", GITHUB_URL.strip(), str(WORKSPACE_DIR)], check=True)


In [ ]:
# Real Environment Installation (Requirements 2, 7 & 8)
# Uses uv to create Python 3.10 virtual environment at /content/ns-env
import os
import shutil
import subprocess
import sys
from pathlib import Path

VENV_PATH = Path('/content/ns-env') if Path('/content').exists() else Path('./ns-env')
ENV_BIN = VENV_PATH / 'bin'
ENV_PYTHON = str(ENV_BIN / 'python')
ENV_PIP = str(ENV_BIN / 'pip')

print(f"=== [SETTING UP PYTHON 3.10 VIRTUAL ENVIRONMENT AT {VENV_PATH}] ===")

# 1. Install uv if not installed
if not shutil.which("uv"):
    print("Installing uv package manager...")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)

# 2. Create Python 3.10 venv at VENV_PATH
if not (ENV_BIN / "python").exists():
    print(f"Creating Python 3.10 virtual environment at {VENV_PATH}...")
    subprocess.run(["uv", "venv", str(VENV_PATH), "--python", "3.10"], check=True)

# Prepend virtual environment bin to PATH for all tool calls
os.environ["PATH"] = f"{ENV_BIN}:{os.environ.get('PATH', '')}"

# 3. Install system packages via apt (Requirement 8: Do not hide apt output; capture it)
print("Installing system packages via apt...")
apt_update = subprocess.run(
    ["apt-get", "update", "-y"],
    capture_output=True,
    text=True,
    check=False
)
apt_install = subprocess.run(
    ["apt-get", "install", "-y", "colmap", "ffmpeg", "ninja-build", "libopenblas-dev"],
    capture_output=True,
    text=True,
    check=False
)

APT_FAILURES_LAST_20 = []
if apt_update.returncode != 0:
    print(f"Warning: apt-get update failed with exit code {apt_update.returncode}")
    APT_FAILURES_LAST_20.extend((apt_update.stdout + "\n" + apt_update.stderr).splitlines()[-20:])
if apt_install.returncode != 0:
    print(f"Warning: apt-get install failed with exit code {apt_install.returncode}")
    APT_FAILURES_LAST_20.extend((apt_install.stdout + "\n" + apt_install.stderr).splitlines()[-20:])
if APT_FAILURES_LAST_20:
    APT_FAILURES_LAST_20 = APT_FAILURES_LAST_20[-20:]
    print(f"Captured {len(APT_FAILURES_LAST_20)} lines of apt error output for smoke_report.json.")

# 4. Install pinned dependencies into /content/ns-env
print("Installing PyTorch (cu118) and Nerfstudio 1.1.5 into virtual environment...")
subprocess.run([
    "uv", "pip", "install", "--python", ENV_PYTHON,
    "torch==2.1.2+cu118", "torchvision==0.16.2+cu118",
    "--extra-index-url", "https://download.pytorch.org/whl/cu118"
], check=True)

subprocess.run([
    "uv", "pip", "install", "--python", ENV_PYTHON,
    "nerfstudio==1.1.5"
], check=True)

# 5. Verify torch and CUDA with ENV_PYTHON (Requirement 7: fail if CUDA is unavailable)
print("Verifying PyTorch and CUDA availability via ENV_PYTHON...")
cuda_check_code = (
    "import torch\n"
    "if not torch.cuda.is_available():\n"
    "    raise SystemExit('CUDA is not available in torch')\n"
    "print(f'{torch.__version__},{torch.version.cuda},{torch.cuda.get_device_name(0)}')\n"
)
cuda_check_res = subprocess.run(
    [ENV_PYTHON, "-c", cuda_check_code],
    capture_output=True,
    text=True,
    check=False
)
if cuda_check_res.returncode != 0:
    err_text = (cuda_check_res.stderr or cuda_check_res.stdout).strip()
    raise SystemExit(
        f"FATAL ERROR: CUDA is unavailable in {ENV_PYTHON} after environment setup.\n"
        f"Details: {err_text}"
    )

torch_parts = cuda_check_res.stdout.strip().split(",")
print(f"Verified {ENV_PYTHON}: PyTorch {torch_parts[0]}, CUDA {torch_parts[1]}, Device: {torch_parts[2]}")


In [ ]:
# Pure Helper Functions & Execution Helper (Requirements 1, 2, 4, 6, 9)
import json
import os
import re
import subprocess
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

def parse_colmap_cuda_support(header_or_output: str) -> bool:
    """
    Parses COLMAP version header or help output to determine CUDA support.
    Requirement 1: 'without CUDA' means no CUDA, 'with CUDA' means CUDA.
    Do not test for the substring 'CUDA' alone.
    """
    if not header_or_output:
        return False
    lower = header_or_output.lower()
    if "without cuda" in lower:
        return False
    if "with cuda" in lower:
        return True
    return False

def build_colmap_step_commands(
    database_path: Path,
    image_path: Path,
    sparse_path: Path,
    colmap_cuda_support: bool,
    feat_help: str = "",
    match_help: str = "",
) -> Tuple[List[str], List[str], List[str]]:
    """
    Builds feature_extractor, exhaustive_matcher, and mapper commands.
    Requirement 2: If COLMAP has no CUDA, add CPU options confirmed from help text.
    """
    feat_cmd = [
        "colmap", "feature_extractor",
        "--database_path", str(database_path),
        "--image_path", str(image_path),
        "--ImageReader.camera_model", "SIMPLE_RADIAL",
        "--ImageReader.single_camera", "1",
    ]
    match_cmd = [
        "colmap", "exhaustive_matcher",
        "--database_path", str(database_path),
    ]
    map_cmd = [
        "colmap", "mapper",
        "--database_path", str(database_path),
        "--image_path", str(image_path),
        "--output_path", str(sparse_path),
    ]

    if not colmap_cuda_support:
        # Exact flag names confirmed from colmap --help
        if not feat_help or "--SiftExtraction.use_gpu" in feat_help:
            feat_cmd.extend(["--SiftExtraction.use_gpu", "0"])
        if not match_help or "--SiftMatching.use_gpu" in match_help:
            match_cmd.extend(["--SiftMatching.use_gpu", "0"])

    return feat_cmd, match_cmd, map_cmd

def extract_usage_lines(help_text: str) -> List[str]:
    """Extracts usage lines from help text for reporting on structure failure."""
    lines = help_text.splitlines()
    usage: List[str] = []
    capture = False
    for line in lines:
        stripped = line.strip()
        if stripped.lower().startswith("usage:"):
            capture = True
            usage.append(stripped)
        elif capture:
            if not stripped or stripped.startswith("╭─") or stripped.startswith("Options:") or stripped.startswith("-"):
                break
            usage.append(stripped)
    return usage

def parse_subcommands_list(help_text: str) -> List[str]:
    """
    Parses available subcommand names from help text without line-equality tests.
    Parses {cmd1,cmd2,...} patterns and subcommands sections.
    """
    subcommands: List[str] = []
    # Match {subcmd1,subcmd2,...}
    curly_matches = re.findall(r"\{([a-zA-Z0-9_\-,\s]+)\}", help_text)
    for match in curly_matches:
        for p in match.split(","):
            token = p.strip()
            if token and not token.startswith("-") and token not in subcommands:
                subcommands.append(token)
    if subcommands:
        return subcommands

    # Parse lines under a subcommands block/table
    lines = help_text.splitlines()
    in_section = False
    for line in lines:
        lower = line.lower()
        if "subcommand" in lower:
            in_section = True
            continue
        if in_section:
            if line.startswith("╭─") or line.startswith("╰─") or (line and not line.startswith(" ") and not line.startswith("│")):
                if subcommands:
                    break
            cleaned = line.strip("│ \t")
            if cleaned:
                token = cleaned.split()[0].strip("{}")
                if token and not token.startswith("-") and re.match(r"^[a-zA-Z0-9_\-]+$", token):
                    if token not in subcommands:
                        subcommands.append(token)

    return subcommands

def determine_viewer_exit_flag(splat_help: str) -> Tuple[str, str]:
    """
    Requirement 6: Accept only --viewer.quit-on-train-completion True,
    or an option that truly disables the viewer according to help text.
    Remove --viewer.make-share-url fallback. If neither exists, fail.
    """
    if "--viewer.quit-on-train-completion" in splat_help:
        return ("--viewer.quit-on-train-completion", "True")
    if "--vis" in splat_help:
        if "none" in splat_help.lower():
            return ("--vis", "none")
        return ("--vis", "tensorboard")
    raise ValueError(
        "FATAL: No valid viewer exit option found in ns-train --help. "
        "Required '--viewer.quit-on-train-completion True' or a viewer disabling option (e.g. '--vis')."
    )

def build_ns_train_command(
    splat_help: str,
    colmap_help: str,
    ns_train_bin: str,
    data_dir: Path,
    output_dir: Path,
    max_iterations: int = 2000,
) -> Tuple[List[str], Dict[str, Any]]:
    """
    Requirement 4 & 6: Builds ns-train command strictly from saved full help text.
    - All trainer options (--data, --output-dir, --max-num-iterations, viewer, pipeline) first,
      in the exact relative order shown in splat_help.
    - Dataparser subcommand name next (e.g. colmap), parsed from subcommand list.
    - Dataparser options next, in the exact relative order shown in colmap_help.
    - If structure cannot be determined, raises RuntimeError with usage lines.
    """
    # 1. Parse subcommands list
    subcommands = parse_subcommands_list(splat_help)
    selected_subcmd = None
    if "colmap" in subcommands:
        selected_subcmd = "colmap"
    elif "nerfstudio-data" in subcommands:
        selected_subcmd = "nerfstudio-data"

    if not selected_subcmd:
        usage = extract_usage_lines(splat_help)
        err_msg = (
            f"FATAL: Could not determine dataparser subcommand from ns-train splatfacto --help. "
            f"Discovered subcommands: {subcommands}. Usage lines: {' '.join(usage)}"
        )
        raise RuntimeError(err_msg)

    # 2. Viewer exit option
    viewer_flag, viewer_val = determine_viewer_exit_flag(splat_help)

    # 3. Collect trainer options
    trainer_candidates: List[Tuple[str, str]] = [
        ("--data", str(data_dir)),
        ("--output-dir", str(output_dir)),
        ("--max-num-iterations", str(max_iterations)),
        (viewer_flag, viewer_val),
    ]

    # Pipeline model options
    if "--pipeline.model.disable-auto-scale" in splat_help:
        trainer_candidates.append(("--pipeline.model.disable-auto-scale", "True"))
    elif "--disable-auto-scale" in splat_help:
        trainer_candidates.append(("--disable-auto-scale", "True"))

    if "--pipeline.datamanager.camera-optimizer.mode" in splat_help:
        trainer_candidates.append(("--pipeline.datamanager.camera-optimizer.mode", "off"))

    # Verify trainer options are present in splat_help and sort by appearance order
    trainer_options: List[Tuple[str, str]] = []
    for flag, val in trainer_candidates:
        idx = splat_help.find(flag)
        if idx == -1:
            raise RuntimeError(f"FATAL: Expected trainer flag '{flag}' not found in ns-train splatfacto --help")
        trainer_options.append((flag, val))

    trainer_options.sort(key=lambda item: splat_help.find(item[0]))

    # 4. Dataparser options (from colmap_help)
    combined_dp_help = colmap_help if colmap_help else splat_help
    dp_candidates: List[Tuple[str, str]] = []

    # Orientation method (required: none)
    if "--pipeline.datamanager.dataparser.orientation-method" in combined_dp_help:
        dp_candidates.append(("--pipeline.datamanager.dataparser.orientation-method", "none"))
    elif "--orientation-method" in combined_dp_help:
        dp_candidates.append(("--orientation-method", "none"))
    else:
        raise RuntimeError("FATAL: Required orientation-method flag not found in dataparser help text")

    # Center method (required: none)
    if "--pipeline.datamanager.dataparser.center-method" in combined_dp_help:
        dp_candidates.append(("--pipeline.datamanager.dataparser.center-method", "none"))
    elif "--center-method" in combined_dp_help:
        dp_candidates.append(("--center-method", "none"))
    else:
        raise RuntimeError("FATAL: Required center-method flag not found in dataparser help text")

    # Auto-scale poses (required: False)
    if "--pipeline.datamanager.dataparser.auto-scale-poses" in combined_dp_help:
        dp_candidates.append(("--pipeline.datamanager.dataparser.auto-scale-poses", "False"))
    elif "--auto-scale-poses" in combined_dp_help:
        dp_candidates.append(("--auto-scale-poses", "False"))
    else:
        raise RuntimeError("FATAL: Required auto-scale-poses flag not found in dataparser help text")

    # Sort dataparser options by appearance order in combined_dp_help
    dp_candidates.sort(key=lambda item: combined_dp_help.find(item[0]))

    # 5. Assemble final command:
    # All trainer options first, then subcommand name, then dataparser options
    cmd = [ns_train_bin, "splatfacto"]
    for flag, val in trainer_options:
        cmd.extend([flag, val])
    cmd.append(selected_subcmd)
    for flag, val in dp_candidates:
        cmd.extend([flag, val])

    metadata = {
        "subcommand": selected_subcmd,
        "trainer_options": trainer_options,
        "dataparser_options": dp_candidates,
    }
    return cmd, metadata

def run_command_checked(
    cmd: List[str],
    step_name: str,
    expected_outputs: Optional[List[Path]] = None,
    timeout_s: int = 600,
) -> Tuple[bool, Dict[str, Any]]:
    """
    Executes command via subprocess.run(..., check=False).
    Verifies exit code == 0 AND all expected output files exist and are non-empty.
    Returns real duration and last 40 lines of output on failure.
    """
    t_start = time.time()
    print(f"\n[CMD: {step_name}] Running: {' '.join(cmd)}")

    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_s, check=False)
        duration = round(time.time() - t_start, 2)
        stdout_lines = proc.stdout.splitlines() if proc.stdout else []
        stderr_lines = proc.stderr.splitlines() if proc.stderr else []
        all_output_lines = stdout_lines + stderr_lines

        code_ok = (proc.returncode == 0)
        missing_files = []
        if expected_outputs:
            for p in expected_outputs:
                if not p.exists() or (p.is_file() and p.stat().st_size == 0):
                    missing_files.append(str(p))

        files_ok = (len(missing_files) == 0)
        passed = code_ok and files_ok
        
        result_info = {
            "step": step_name,
            "command": cmd,
            "passed": passed,
            "returncode": proc.returncode,
            "duration_s": duration,
            "missing_files": missing_files,
            "last_output_lines": all_output_lines[-40:] if not passed else all_output_lines[-10:],
        }

        if passed:
            print(f"[{step_name}] PASSED in {duration}s")
        else:
            print(f"[{step_name}] FAILED in {duration}s (exit_code={proc.returncode}, missing_files={missing_files})")
            print("--- Last 20 lines of output ---")
            for line in all_output_lines[-20:]:
                print(line)
            print("-------------------------------")

        return passed, result_info

    except FileNotFoundError as e:
        duration = round(time.time() - t_start, 2)
        err_msg = f"Tool executable '{cmd[0]}' not found: {e}"
        print(f"[{step_name}] FAILED in {duration}s: {err_msg}")
        return False, {
            "step": step_name,
            "command": cmd,
            "passed": False,
            "returncode": -1,
            "duration_s": duration,
            "error": err_msg,
            "last_output_lines": [err_msg],
        }
    except subprocess.TimeoutExpired as e:
        duration = round(time.time() - t_start, 2)
        err_msg = f"Command timed out after {timeout_s}s"
        print(f"[{step_name}] FAILED in {duration}s: {err_msg}")
        return False, {
            "step": step_name,
            "command": cmd,
            "passed": False,
            "returncode": -1,
            "duration_s": duration,
            "error": err_msg,
            "last_output_lines": [err_msg],
        }


In [ ]:
# Step 0 Probe: Hardware, Compilers, Python, COLMAP CUDA Audit, and Full Help Discovery
import json
import time
import subprocess
import sys

smoke_report = {
    "schema_version": 1,
    "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "environment_probes": {},
    "tool_help_outputs": {},
    "steps": {},
}

# Attach any apt failure output captured in install step (Requirement 8)
if "APT_FAILURES_LAST_20" in globals() and APT_FAILURES_LAST_20:
    smoke_report["environment_probes"]["apt_failure_last_20_lines"] = APT_FAILURES_LAST_20

print("=== [STEP 0: ENVIRONMENT & TOOL PROBES] ===")
# 1. Python version probe from ns-env
env_py_probe = subprocess.run([ENV_PYTHON, "--version"], capture_output=True, text=True, check=False)
py_ver = env_py_probe.stdout.strip() if env_py_probe.returncode == 0 else sys.version.split()[0]
print(f"Python Version: {py_ver}")
smoke_report["environment_probes"]["python"] = py_ver

# 2. PyTorch & CUDA probe
torch_check_code = "import torch; print(f'{torch.__version__},{torch.cuda.is_available()},{torch.version.cuda},{torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}')"
torch_res = subprocess.run([ENV_PYTHON, "-c", torch_check_code], capture_output=True, text=True, check=False)
if torch_res.returncode == 0:
    t_parts = torch_res.stdout.strip().split(",")
    t_ver, t_cuda_avail, t_cuda_ver, t_dev_name = t_parts[0], (t_parts[1] == "True"), t_parts[2], t_parts[3]
else:
    t_ver, t_cuda_avail, t_cuda_ver, t_dev_name = "unknown", False, "unknown", "unknown"

print(f"PyTorch: {t_ver}, CUDA Available: {t_cuda_avail}, CUDA Version: {t_cuda_ver}, Device: {t_dev_name}")
smoke_report["environment_probes"]["torch"] = t_ver
smoke_report["environment_probes"]["cuda_available"] = t_cuda_avail
smoke_report["environment_probes"]["cuda_version"] = t_cuda_ver
smoke_report["environment_probes"]["gpu_device"] = t_dev_name

# 3. nvcc compiler probe
nvcc_res = subprocess.run(["nvcc", "--version"], capture_output=True, text=True, check=False)
nvcc_info = nvcc_res.stdout.splitlines()[-1] if nvcc_res.returncode == 0 else "nvcc not found"
print(f"NVCC: {nvcc_info}")
smoke_report["environment_probes"]["nvcc"] = nvcc_info

# 4. ninja build tool probe
ninja_res = subprocess.run(["ninja", "--version"], capture_output=True, text=True, check=False)
ninja_info = ninja_res.stdout.strip() if ninja_res.returncode == 0 else "ninja not found"
print(f"Ninja: {ninja_info}")
smoke_report["environment_probes"]["ninja"] = ninja_info

# 5. COLMAP CUDA Probe (Requirements 1 & 2)
colmap_probe = subprocess.run(["colmap", "-h"], capture_output=True, text=True, check=False)
has_colmap = (colmap_probe.returncode == 0)
colmap_header = colmap_probe.stdout.splitlines()[0] if colmap_probe.stdout else ""
colmap_cuda_support = parse_colmap_cuda_support(colmap_probe.stdout)

print(f"COLMAP installed: {colmap_header} (CUDA Support: {colmap_cuda_support})")
smoke_report["environment_probes"]["colmap_installed"] = has_colmap
smoke_report["environment_probes"]["colmap_header"] = colmap_header
smoke_report["environment_probes"]["colmap_cuda_support"] = colmap_cuda_support

# 6. Save FULL Help Outputs into smoke_report.json and separate files on Drive (Requirement 3, 4 & 6)
help_dir = DRIVE_RUN_DIR / "help_outputs"
help_dir.mkdir(parents=True, exist_ok=True)

tools_to_probe = [
    ([str(ENV_BIN / "ns-train"), "splatfacto", "--help"], "ns_train_splatfacto"),
    ([str(ENV_BIN / "ns-train"), "splatfacto", "colmap", "--help"], "ns_train_splatfacto_colmap"),
    ([str(ENV_BIN / "ns-export"), "gaussian-splat", "--help"], "ns_export_gaussian_splat"),
    (["colmap", "feature_extractor", "--help"], "colmap_feature_extractor"),
    (["colmap", "exhaustive_matcher", "--help"], "colmap_exhaustive_matcher"),
]

for tool_cmd, key in tools_to_probe:
    try:
        help_res = subprocess.run(tool_cmd, capture_output=True, text=True, check=False)
        if help_res.returncode == 0:
            full_help = help_res.stdout
            smoke_report["tool_help_outputs"][key] = full_help
            help_file = help_dir / f"{key}.txt"
            help_file.write_text(full_help, encoding="utf-8")
            print(f"Discovered and saved FULL help for: {' '.join(tool_cmd[:2])} ({len(full_help)} chars) to {help_file}")
        else:
            smoke_report["tool_help_outputs"][key] = f"Failed with code {help_res.returncode}"
    except FileNotFoundError as e:
        smoke_report["tool_help_outputs"][key] = f"Tool not found: {e}"


In [ ]:
# Execute Smoke Test Steps 1 to 4 with Subprocess Wrappers & File Verifications
# Wrapped in try/finally to GUARANTEE smoke_report.json is written even after a crash (Requirement 1)
import json
import os
import shutil
import time
from pathlib import Path

SMOKE_DIR = DRIVE_RUN_DIR / "smoke_test_run"
SMOKE_DIR.mkdir(parents=True, exist_ok=True)
report_json_path = DRIVE_RUN_DIR / "smoke_report.json"

try:
    # --- Step 1: Environment Assertion ---
    if not smoke_report["environment_probes"].get("cuda_available"):
        smoke_report["steps"]["step_1_env"] = {"passed": False, "error": "No CUDA GPU attached"}
        raise RuntimeError("No GPU attached. Halting smoke test.")
    else:
        smoke_report["steps"]["step_1_env"] = {
            "passed": True,
            "device": smoke_report["environment_probes"].get("gpu_device"),
        }
        print("Step 1/4 (GPU Environment) PASSED")

    # --- Step 2: COLMAP Sparse Reconstruction (Requirement 2) ---
    colmap_out_dir = SMOKE_DIR / "colmap"
    colmap_out_dir.mkdir(parents=True, exist_ok=True)
    colmap_db = colmap_out_dir / "database.db"
    colmap_sparse = colmap_out_dir / "sparse" / "0"
    colmap_out_dir_sparse = colmap_out_dir / "sparse"
    colmap_out_dir_sparse.mkdir(parents=True, exist_ok=True)

    colmap_cuda_support = smoke_report["environment_probes"].get("colmap_cuda_support", False)
    if not colmap_cuda_support:
        os.environ["QT_QPA_PLATFORM"] = "offscreen"
        print("[COLMAP] Running in CPU mode (QT_QPA_PLATFORM=offscreen, CPU feature extraction/matching)")
    else:
        print("[COLMAP] Running in CUDA mode")

    feat_help_text = smoke_report["tool_help_outputs"].get("colmap_feature_extractor", "")
    match_help_text = smoke_report["tool_help_outputs"].get("colmap_exhaustive_matcher", "")

    feat_cmd, match_cmd, map_cmd = build_colmap_step_commands(
        database_path=colmap_db,
        image_path=WORKING_FRAMES_DIR,
        sparse_path=colmap_out_dir_sparse,
        colmap_cuda_support=colmap_cuda_support,
        feat_help=feat_help_text,
        match_help=match_help_text,
    )

    t_colmap_start = time.time()
    ok_feat, info_feat = run_command_checked(
        feat_cmd,
        "colmap_feature_extractor",
        expected_outputs=[colmap_db],
        timeout_s=TIMEOUT_COLMAP_FEATURE_S,
    )

    if ok_feat:
        ok_match, info_match = run_command_checked(
            match_cmd,
            "colmap_exhaustive_matcher",
            expected_outputs=[colmap_db],
            timeout_s=TIMEOUT_COLMAP_MATCH_S,
        )
    else:
        ok_match, info_match = False, {"passed": False, "error": "Skipped due to feature extraction failure"}

    if ok_match:
        ok_map, info_map = run_command_checked(
            map_cmd,
            "colmap_mapper",
            timeout_s=TIMEOUT_COLMAP_MAP_S,
        )
        sparse_valid = (
            (colmap_sparse / "cameras.bin").is_file() or (colmap_sparse / "cameras.txt").is_file()
        ) and (
            (colmap_sparse / "images.bin").is_file() or (colmap_sparse / "images.txt").is_file()
        )
        if not sparse_valid:
            ok_map = False
            info_map["passed"] = False
            info_map["missing_files"] = [str(colmap_sparse / "cameras.bin/txt"), str(colmap_sparse / "images.bin/txt")]
    else:
        ok_map, info_map = False, {"passed": False, "error": "Skipped due to matching failure"}

    colmap_duration = round(time.time() - t_colmap_start, 2)
    step_2_passed = ok_feat and ok_match and ok_map
    smoke_report["steps"]["step_2_colmap"] = {
        "passed": step_2_passed,
        "colmap_path": "cuda" if colmap_cuda_support else "cpu",
        "duration_s": colmap_duration,
        "feature": info_feat,
        "match": info_match,
        "map": info_map,
    }

    if not step_2_passed:
        raise RuntimeError("Step 2 (COLMAP reconstruction) failed. Halting before training.")

    # --- Data Layout Preparation for Nerfstudio ---
    ns_data_dir = SMOKE_DIR / "nerfstudio_data"
    ns_data_dir.mkdir(parents=True, exist_ok=True)
    ns_images_dir = ns_data_dir / "images"
    ns_colmap_sparse = ns_data_dir / "colmap" / "sparse" / "0"
    ns_sparse_direct = ns_data_dir / "sparse" / "0"

    if ns_images_dir.exists():
        if ns_images_dir.is_symlink():
            ns_images_dir.unlink()
        else:
            shutil.rmtree(ns_images_dir)
    try:
        ns_images_dir.symlink_to(WORKING_FRAMES_DIR, target_is_directory=True)
    except Exception:
        shutil.copytree(WORKING_FRAMES_DIR, ns_images_dir)

    ns_colmap_sparse.parent.mkdir(parents=True, exist_ok=True)
    if ns_colmap_sparse.exists():
        if ns_colmap_sparse.is_symlink():
            ns_colmap_sparse.unlink()
        else:
            shutil.rmtree(ns_colmap_sparse)
    try:
        ns_colmap_sparse.symlink_to(colmap_sparse, target_is_directory=True)
    except Exception:
        shutil.copytree(colmap_sparse, ns_colmap_sparse)

    ns_sparse_direct.parent.mkdir(parents=True, exist_ok=True)
    if not ns_sparse_direct.exists():
        try:
            ns_sparse_direct.symlink_to(ns_colmap_sparse, target_is_directory=True)
        except Exception:
            pass

    splat_help = smoke_report["tool_help_outputs"].get("ns_train_splatfacto", "")
    colmap_help = smoke_report["tool_help_outputs"].get("ns_train_splatfacto_colmap", "")

    # --- Step 3a: Splatfacto Dry-Run (10 iterations) (Requirement 5) ---
    train_out_dir_3a = SMOKE_DIR / "nerfstudio_models_3a"
    train_out_dir_3a.mkdir(parents=True, exist_ok=True)

    try:
        cmd_3a, meta_3a = build_ns_train_command(
            splat_help=splat_help,
            colmap_help=colmap_help,
            ns_train_bin=str(ENV_BIN / "ns-train"),
            data_dir=ns_data_dir,
            output_dir=train_out_dir_3a,
            max_iterations=10,
        )
    except Exception as e:
        usage = extract_usage_lines(splat_help)
        smoke_report["steps"]["step_3a_train"] = {
            "passed": False,
            "error": str(e),
            "usage_lines": usage,
        }
        raise

    ok_train_3a, info_train_3a = run_command_checked(
        cmd_3a,
        "ns_train_splatfacto_3a_dryrun",
        timeout_s=TIMEOUT_TRAIN_3A_S,
    )
    config_files_3a = list(train_out_dir_3a.glob("**/config.yml"))
    train_3a_passed = ok_train_3a and len(config_files_3a) > 0 and config_files_3a[0].stat().st_size > 0
    smoke_report["steps"]["step_3a_train"] = {
        "passed": train_3a_passed,
        "config_file": str(config_files_3a[0]) if config_files_3a else None,
        "details": info_train_3a,
        "command": cmd_3a,
    }

    if not train_3a_passed:
        raise RuntimeError("Step 3a (Nerfstudio dry-run 10-iteration training) failed. Halting before step 3b.")

    print(f"Step 3a PASSED. config.yml generated successfully at: {config_files_3a[0]}")

    # --- Step 3b: Splatfacto Training (2000 iterations) (Requirement 5) ---
    train_out_dir_3b = SMOKE_DIR / "nerfstudio_models"
    train_out_dir_3b.mkdir(parents=True, exist_ok=True)

    cmd_3b, meta_3b = build_ns_train_command(
        splat_help=splat_help,
        colmap_help=colmap_help,
        ns_train_bin=str(ENV_BIN / "ns-train"),
        data_dir=ns_data_dir,
        output_dir=train_out_dir_3b,
        max_iterations=2000,
    )

    ok_train_3b, info_train_3b = run_command_checked(
        cmd_3b,
        "ns_train_splatfacto_3b_full",
        timeout_s=TIMEOUT_TRAIN_3B_S,
    )
    config_files_3b = list(train_out_dir_3b.glob("**/config.yml"))
    train_3b_passed = ok_train_3b and len(config_files_3b) > 0 and config_files_3b[0].stat().st_size > 0
    smoke_report["steps"]["step_3b_train"] = {
        "passed": train_3b_passed,
        "config_file": str(config_files_3b[0]) if config_files_3b else None,
        "details": info_train_3b,
        "command": cmd_3b,
    }

    if not train_3b_passed:
        raise RuntimeError("Step 3b (Nerfstudio 2000-iteration training) failed. Halting before export.")

    # --- Step 4: Gaussian Splat Export ---
    export_out_dir = SMOKE_DIR / "exports"
    export_out_dir.mkdir(parents=True, exist_ok=True)
    exported_ply = export_out_dir / "splat.ply"

    export_cmd = [
        str(ENV_BIN / "ns-export"), "gaussian-splat",
        "--load-config", str(config_files_3b[0]),
        "--output-dir", str(export_out_dir),
    ]
    ok_export, info_export = run_command_checked(
        export_cmd,
        "ns_export_gaussian_splat",
        expected_outputs=[exported_ply],
        timeout_s=TIMEOUT_EXPORT_S,
    )
    export_passed = ok_export and exported_ply.is_file() and exported_ply.stat().st_size > 0
    smoke_report["steps"]["step_4_export"] = {
        "passed": export_passed,
        "ply_file": str(exported_ply) if export_passed else None,
        "details": info_export,
    }

finally:
    # Requirement 1 & 8: Guarantee smoke_report.json is written even on error, including apt errors
    if "APT_FAILURES_LAST_20" in globals() and APT_FAILURES_LAST_20:
        smoke_report.setdefault("environment_probes", {})["apt_failure_last_20_lines"] = APT_FAILURES_LAST_20
    with open(report_json_path, "w", encoding="utf-8") as f:
        json.dump(smoke_report, f, indent=2)
    print(f"\n[REPORT WRITTEN] Smoke report successfully saved to: {report_json_path}")
    print(json.dumps(smoke_report.get("steps", {}), indent=2))
